In [ ]:
import datetime
import numpy as np
np.set_printoptions(precision=4)
from numpy.typing import NDArray
from typing import Optional
import pandas as pd

sym = 'NVDA'

date = f'{datetime.datetime.now()+datetime.timedelta(days=-1):%Y%m%d}'
filename = f'{sym}_5s_{date}.csv'
load_from_csv = True
bars = None
if load_from_csv:
    bars_df = pd.read_csv(rf'.\data\{filename}', parse_dates=['date'])
if 'open_' in bars_df.columns:
    bars_df.rename(columns={'open_': 'open'}, inplace=True)
bars_df['log_avg'] = np.log(bars_df['average'])
bars_df[['open', 'high', 'low', 'close', 'average', 'date']].head()

https://github.com/rfrenoy/psquare

In [ ]:
from collections import deque

class PSquareQuantileEstimator:
    def __init__(self, quantiles):
        self.quantiles = np.sort(quantiles)
        self.n = 0
        self.q = [0] * len(quantiles)
        self.dn = [0] * len(quantiles)
        self.np = [0] * len(quantiles)
        self.n1 = [0] * len(quantiles)
        self.n2 = [0] * len(quantiles)

    def update(self, x):
        self.n += 1
        if self.n <= len(self.quantiles):
            self.q[self.n - 1] = x
            if self.n == len(self.quantiles):
                self.q.sort()
        else:
            k = self.binary_search(x)
            for i in range(len(self.quantiles)):
                if i < k:
                    self.np[i] += self.quantiles[i]
                elif i > k:
                    self.n1[i] += 1
                else:
                    self.np[i] += self.quantiles[i] - 1
                    self.n1[i] += 1
            self.adjust()

    def binary_search(self, x):
        left, right = 0, len(self.q) - 1
        while left < right:
            mid = (left + right) // 2
            if self.q[mid] <= x:
                left = mid + 1
            else:
                right = mid
        return left

    def adjust(self):
        for i in range(1, len(self.quantiles) - 1):
            d = self.np[i] - self.n1[i]
            if (d >= 1 and self.n2[i] > 1) or (d <= -1 and self.n1[i] > 1):
                d = int(np.sign(d))
                qs = self.parabolic(i, d)
                if self.q[i-1] < qs < self.q[i+1]:
                    self.q[i] = qs
                else:
                    self.q[i] = self.linear(i, d)
                self.n1[i] += d
                self.n2[i] -= d

    def parabolic(self, i, d):
        return self.q[i] + d / (self.n2[i] - self.n1[i]) * (
            (self.n2[i] - self.n1[i] + d) * (self.q[i+1] - self.q[i]) / (self.n2[i+1] - self.n1[i+1]) +
            (self.n1[i] - self.n2[i] - d) * (self.q[i] - self.q[i-1]) / (self.n2[i-1] - self.n1[i-1])
        ) / 2

    def linear(self, i, d):
        return self.q[i] + d * (self.q[i+d] - self.q[i]) / (self.n2[i+d] - self.n1[i])

    def get_quantiles(self):
        return self.q

class AdaptiveHistogram:
    def __init__(self, num_bins=10, window_size=100):
        self.num_bins = num_bins
        self.window_size = window_size
        self.data_window = deque(maxlen=window_size)
        self.bins = [[] for _ in range(num_bins)]
        self.quantile_estimator = PSquareQuantileEstimator(np.linspace(0, 1, num_bins + 1))
        self.min_value = float('inf')
        self.max_value = float('-inf')

    def update(self, value):
        self.data_window.append(value)
        self.quantile_estimator.update(value)
        
        if value < self.min_value or value > self.max_value:
            self.min_value = min(self.min_value, value)
            self.max_value = max(self.max_value, value)
            return True  # Flag as anomaly
        
        bin_index = self.get_bin_index(value)
        self.bins[bin_index].append(value)
        
        if len(self.data_window) % (self.window_size // 10) == 0:
            self.adjust_bins()
        
        return False

    def get_bin_index(self, value):
        quantiles = self.quantile_estimator.get_quantiles()
        return np.searchsorted(quantiles, value) - 1

    def adjust_bins(self):
        self.bins = [[] for _ in range(self.num_bins)]
        quantiles = self.quantile_estimator.get_quantiles()
        for value in self.data_window:
            bin_index = np.searchsorted(quantiles, value) - 1
            self.bins[bin_index].append(value)

class OHLCAnomalyDetector:
    def __init__(self, num_bins=100, window_size=1000):
        self.histograms = [AdaptiveHistogram(num_bins, window_size) for _ in range(4)]
        self.labels = ['Open', 'High', 'Low', 'Close']

    def detect_anomaly(self, ohlc_data):
        anomalies = []
        for i, value in enumerate(ohlc_data):
            if self.histograms[i].update(value):
                anomalies.append(self.labels[i])
        return anomalies

# Example usage
detector = OHLCAnomalyDetector()

def process_data_point(ohlc_data, index):
    anomalies = detector.detect_anomaly(ohlc_data)
    if anomalies:
        print(f"Anomaly detected in: {', '.join(anomalies)}")
        print(f"Data point: {ohlc_data}")
    return bool(anomalies)


In [ ]:
import random
import time

prev_data_point = None
for row in bars_df[['open', 'high', 'low', 'close', 'date']].set_index('date').itertuples(index=True, name=None):
    # print(f"Processed data point: {row}")
    index, *data_point = row
    process_data_point(data_point, index)
    


In [ ]:
# Simulating data stream
import random
import time

while True:
    # Generate sample data (replace this with your actual data stream)
    base_price = random.uniform(50, 150)
    ohlc_data = (
        base_price + random.uniform(-1, 1),
        base_price + random.uniform(0, 2),
        base_price - random.uniform(0, 2),
        base_price + random.uniform(-1, 1)
    )
    
    # Occasionally introduce an anomaly
    if random.random() < 0.05:
        index = random.randint(0, 3)
        ohlc_data = list(ohlc_data)
        ohlc_data[index] *= random.uniform(1.5, 2.0)  # Increase one value significantly
        ohlc_data = tuple(ohlc_data)

    process_data_point(ohlc_data)
    time.sleep(5)  # Wait for 5 seconds before the next data point
